In [1]:
import os
import sys

# Path to DSLPT repository
repo_path = "/home/jocareher/Downloads/DSLPT"

# Add DSLPT repository to python path
sys.path.append(repo_path)

import cv2
import torch
import numpy as np
from tqdm import tqdm
from model import Dynamic_sparse_alignment_network

from Config.default import _C as cfg

In [ ]:

def preprocess_image(image: np.ndarray) -> torch.Tensor:
    """
    Resize and normalize an input image for model inference.

    Args:
        image: Input image in BGR format with shape (H, W, 3).

    Returns:
        A tensor with shape (1, 3, 256, 256).
    """
    resized_image = cv2.resize(image, (256, 256))
    normalized_image = resized_image.astype(np.float32) / 255.0
    tensor_image = torch.tensor(normalized_image, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0)
    return tensor_image


def filter_68_landmarks(landmarks: np.ndarray) -> np.ndarray:
    """
    Extract the 68 landmark subset using the predefined mapping indices.

    Args:
        landmarks: Landmark predictions from the model.

    Returns:
        A NumPy array of shape (68, 2) with normalized landmark coordinates.
    """
    mapping_indices = [
        0, 2, 3, 4, 7, 9, 12, 14, 16, 18, 20, 22, 25, 27, 29, 31, 32,
        33, 34, 35, 36, 37, 42, 43, 44, 45, 46, 51, 52, 53, 54, 55,
        56, 57, 58, 59, 60, 61, 63, 64, 65, 67, 68, 69, 71, 72, 73,
        75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89,
        90, 91, 92, 93, 94, 95
    ]
    final_landmarks = landmarks[-1]
    return final_landmarks[mapping_indices, :]


def get_landmark_groups_68() -> List[List[int]]:
    """
    Return the standard connectivity groups for 68 facial landmarks.

    Returns:
        A list of landmark index groups describing the facial topology.
    """
    return [
        list(range(0, 17)),                         # Jaw
        list(range(17, 22)),                        # Right eyebrow
        list(range(22, 27)),                        # Left eyebrow
        list(range(27, 31)),                        # Nose bridge
        list(range(31, 36)),                        # Nose base
        [36, 37, 38, 39, 40, 41, 36],               # Right eye
        [42, 43, 44, 45, 46, 47, 42],               # Left eye
        [48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 48],  # Outer lip
        [60, 61, 62, 63, 64, 65, 66, 67, 60],       # Inner lip
    ]


def draw_landmark_connections(
    image: np.ndarray,
    landmarks_px: np.ndarray,
    color: Tuple[int, int, int],
    line_thickness: int,
) -> None:
    """
    Draw connection lines between 68 facial landmarks.

    Args:
        image: Input image in BGR format.
        landmarks_px: Landmark coordinates in absolute pixel space with shape (68, 2).
        color: BGR color for the connection lines.
        line_thickness: Thickness of the connection lines.
    """
    groups = get_landmark_groups_68()

    for group in groups:
        for start_idx, end_idx in zip(group[:-1], group[1:]):
            x1, y1 = landmarks_px[start_idx]
            x2, y2 = landmarks_px[end_idx]

            cv2.line(
                image,
                (int(round(x1)), int(round(y1))),
                (int(round(x2)), int(round(y2))),
                color,
                line_thickness,
            )


def draw_landmarks(
    image: np.ndarray,
    landmarks: np.ndarray,
    color: Tuple[int, int, int] = (255, 255, 0),
) -> np.ndarray:
    """
    Draw 68 landmarks and their connections on the image.

    Args:
        image: Input image in BGR format.
        landmarks: Landmark coordinates normalized to [0, 1] with shape (68, 2).
        color: BGR color for both landmarks and connection lines.
            Default is cyan in OpenCV BGR format.

    Returns:
        The image with landmarks and connections drawn.
    """
    height, width, _ = image.shape
    diagonal = float((width ** 2 + height ** 2) ** 0.5)

    radius = max(8, min(int(diagonal * 0.01), 16))
    line_thickness = max(1, int(radius * 0.6))
    point_thickness = -1

    landmarks_px = landmarks.copy().astype(np.float32)
    landmarks_px[:, 0] *= width
    landmarks_px[:, 1] *= height

    draw_landmark_connections(
        image=image,
        landmarks_px=landmarks_px,
        color=color,
        line_thickness=line_thickness,
    )

    for x_coord, y_coord in landmarks_px:
        cv2.circle(
            image,
            (int(round(x_coord)), int(round(y_coord))),
            radius,
            color,
            point_thickness,
        )

    return image


def save_landmarks_to_txt(
    landmarks: np.ndarray,
    output_txt_path: str,
    image_width: int,
    image_height: int,
) -> None:
    """
    Save landmarks to a txt file with one coordinate pair per line.

    Output format:
        x1 y1
        x2 y2
        ...
        x68 y68

    Args:
        landmarks: Landmark coordinates normalized to [0, 1] with shape (68, 2).
        output_txt_path: Path where the txt file will be saved.
        image_width: Original image width in pixels.
        image_height: Original image height in pixels.
    """
    landmarks_px = landmarks.copy().astype(np.float32)
    landmarks_px[:, 0] *= image_width
    landmarks_px[:, 1] *= image_height

    with open(output_txt_path, "w") as file:
        for x_coord, y_coord in landmarks_px:
            file.write(f"{x_coord} {y_coord}\n")


def process_image(
    image_path: str,
    output_image_dir: str,
    output_label_dir: str,
    model,
    device: torch.device,
    landmark_color: Tuple[int, int, int] = (255, 255, 0),
) -> bool:
    """
    Process a single image, run landmark inference, and save outputs.

    Args:
        image_path: Path to the input image.
        output_image_dir: Directory where plotted images will be saved.
        output_label_dir: Directory where label txt files will be saved.
        model: Landmark model.
        device: Torch device used for inference.
        landmark_color: BGR color used for landmarks and lines.

    Returns:
        True if processing succeeded.
    """
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Could not load image: {image_path}")

    height, width, _ = image.shape
    input_tensor = preprocess_image(image).to(device)

    with torch.inference_mode():
        output_list, _, _, _ = model(input_tensor)
        landmarks = output_list[-1].squeeze().cpu().numpy()
        landmarks_68 = filter_68_landmarks(landmarks)

    output_image_path = os.path.join(output_image_dir, os.path.basename(image_path))
    plotted_image = draw_landmarks(
        image=image.copy(),
        landmarks=landmarks_68,
        color=landmark_color,
    )
    cv2.imwrite(output_image_path, plotted_image)

    output_label_path = os.path.join(
        output_label_dir,
        f"{os.path.splitext(os.path.basename(image_path))[0]}.txt",
    )
    save_landmarks_to_txt(
        landmarks=landmarks_68,
        output_txt_path=output_label_path,
        image_width=width,
        image_height=height,
    )

    return True


def process_directory(
    input_dir: str,
    output_dir: str,
    model,
    device: torch.device,
    landmark_color: Tuple[int, int, int] = (255, 255, 0),
) -> None:
    """
    Process a directory of images and save landmark predictions.

    Args:
        input_dir: Directory containing input images.
        output_dir: Directory where outputs will be saved.
        model: Landmark model.
        device: Torch device used for inference.
        landmark_color: BGR color used for landmarks and lines.
    """
    image_output_dir = os.path.join(output_dir, "images")
    label_output_dir = os.path.join(output_dir, "labels")
    os.makedirs(image_output_dir, exist_ok=True)
    os.makedirs(label_output_dir, exist_ok=True)

    images = sorted(
        [file_name for file_name in os.listdir(input_dir) if file_name.lower().endswith((".jpg", ".png"))]
    )

    processed_count = 0
    failed_images: List[str] = []

    print("\nProcessing images...")
    for image_name in tqdm(images, desc="Processing"):
        image_path = os.path.join(input_dir, image_name)

        try:
            process_image(
                image_path=image_path,
                output_image_dir=image_output_dir,
                output_label_dir=label_output_dir,
                model=model,
                device=device,
                landmark_color=landmark_color,
            )
            processed_count += 1
        except Exception as error:
            print(f"Failed to process {image_name}: {error}")
            failed_images.append(image_name)

    print("\n=== Processing Summary ===")
    print(f"Total images processed: {processed_count}")
    print(f"Total images failed: {len(failed_images)}")

    if failed_images:
        print("Failed images:")
        for image_name in failed_images:
            print(f"  {image_name}")


In [3]:
input_directory = '/home/jocareher/Documents/baby_face_72/images'
output_directory = '/home/jocareher/Documents/results_dslpt'

# Load model
model_path = '/home/jocareher/Downloads/DSLPT_WFLW_6_layers.pth'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Dynamic_sparse_alignment_network(num_point=98, d_model=256, trainable=False,
                                            return_interm_layers=False, nhead=8,
                                            feedforward_dim=1024, initial_path='/home/jocareher/Downloads/DSLPT/Config/init_98.npz',
                                            cfg=cfg)
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

# Process directory
process_directory(input_directory, output_directory, model, device)





/tmp/ipykernel_32267/1825572725.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=device))



Processing images...


Processing: 100%|██████████| 311/311 [02:24<00:00,  2.15it/s]


=== Processing Summary ===
Total images processed: 311
Total images failed: 0
